In [0]:
import scipy.stats

daily_df = spark.sql("""
SELECT p.price_date,
       avg(p.pvpc_eur_kwh) AS avg_pvpc_eur_kwh,
       g.generation_mwh    AS wind_generation_mwh
FROM workspace.ree_energy.gold_hourly_prices p
JOIN workspace.ree_energy.gold_daily_generation g
  ON g.generation_date = p.price_date
 AND g.technology = 'Eólica'
WHERE p.price_date >= date_sub(current_date(), 59)
GROUP BY p.price_date, g.generation_mwh
ORDER BY p.price_date
""").toPandas()

r = daily_df['wind_generation_mwh'].corr(daily_df['avg_pvpc_eur_kwh'])
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
    daily_df['wind_generation_mwh'], daily_df['avg_pvpc_eur_kwh'])

# Compare lowest vs highest wind terciles
q_low, q_high = daily_df['wind_generation_mwh'].quantile([1/3, 2/3])
low_wind  = daily_df[daily_df['wind_generation_mwh'] <= q_low]
high_wind = daily_df[daily_df['wind_generation_mwh'] >= q_high]

print(f"Days analysed: {len(daily_df)}")
print(f"Pearson r (wind vs avg PVPC): {r:.3f}   (R² = {r**2:.3f})")
print(f"Linear fit: PVPC ≈ {intercept:.3f} - {abs(slope):.6f} €/kWh per MWh of wind  (p = {p_value:.3f})")
print(f"Low-wind tercile  (n={len(low_wind)}):  avg wind {low_wind['wind_generation_mwh'].mean():,.0f} MWh/day -> avg PVPC {low_wind['avg_pvpc_eur_kwh'].mean():.3f} €/kWh")
print(f"High-wind tercile (n={len(high_wind)}): avg wind {high_wind['wind_generation_mwh'].mean():,.0f} MWh/day -> avg PVPC {high_wind['avg_pvpc_eur_kwh'].mean():.3f} €/kWh")
display(daily_df.head(5))

In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(daily_df['wind_generation_mwh'] / 1000, daily_df['avg_pvpc_eur_kwh'],
           alpha=0.75, color='#1f77b4', edgecolor='white', s=60)

xs = daily_df['wind_generation_mwh'] / 1000
ax.plot(xs, intercept + slope * daily_df['wind_generation_mwh'],
        color='#d62728', linewidth=2,
        label=f'Trend (r = {r:.3f}, p = {p_value:.3f})')

ax.set_xlabel('Wind generation (GWh / day)')
ax.set_ylabel('Avg PVPC price (€/kWh)')
ax.set_title(f'Wind generation vs household electricity price (PVPC) — last 60 days (n={len(daily_df)})')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Conclusion
Over the last 60 days there is a **weak negative association** between daily wind generation and the daily-average PVPC household price: Pearson r ≈ -0.19 (R² ≈ 0.04), p ≈ 0.15, so the slope is not statistically significant. High-wind days are only slightly cheaper on average, so wind alone explains little of the day-to-day price variation — demand, solar output, gas prices and hydro exports dominate. Correlation here is also not causation.